In [0]:
########################################
# Narrated fact_orders build for Lesson 08.
#
# Builds fact_orders by joining orders_bronze to all four dimensions.
# Runs after the learner has built dim_customer, so all four dimensions exist.
########################################

try:
    _catalog = my_catalog  # set by Classroom-Setup-Common (reliable)
except NameError:
    _catalog = spark.sql("SELECT current_catalog()").collect()[0][0]
_schema  = "vintage_audio"


def _line(char: str = "-", width: int = 72) -> None:
    print(char * width, flush=True)


def _section(label: str) -> None:
    print(flush=True)
    _line("=")
    print(f"  {label}", flush=True)
    _line("=")


def _say(text: str) -> None:
    print(text, flush=True)


def _table_exists(catalog: str, schema: str, table: str) -> bool:
    try:
        spark.sql(f"DESCRIBE TABLE `{catalog}`.`{schema}`.`{table}`")
        return True
    except Exception:
        return False


# ----- Prerequisite check -----------------------------------------------------
_required = ["dim_customer", "dim_product", "dim_store", "dim_date", "orders_bronze"]
_missing = [t for t in _required if not _table_exists(_catalog, _schema, t)]

if _missing:
    _line("=")
    _say(f"  WARNING: missing required tables: {', '.join(_missing)}")
    _say("  Make sure you have run the earlier sections of this notebook")
    _say("  (the dimensions build and the dim_customer build) before running")
    _say("  this cell.")
    _line("=")
else:
    # ----- Begin narrated fact build ------------------------------------------
    _section("BUILDING THE FACT TABLE")
    _say("fact_orders is the centerpiece of the star schema. It holds one row")
    _say("per business event (an order), with surrogate keys pointing at each")
    _say("dimension.")
    _say("")
    _say("Building it is a three-step recipe:")
    _say("  1. CREATE TABLE with foreign key columns and FK constraints.")
    _say("  2. INSERT, joining orders_bronze to all four dimensions to")
    _say("     translate natural keys (customer_id, product_id, ...) into")
    _say("     surrogate keys (customer_key, product_key, ...).")
    _say("  3. Verify the row count matches orders_bronze.")

    # Step 1: create the empty fact table
    _section("STEP 1 of 3: Create the fact table with FK constraints")
    _say("The fact table has four foreign key columns, one per dimension.")
    _say("Each is declared NOT NULL because we expect every order to belong")
    _say("to a known customer, product, store, and date.")
    _say("")
    _say("Each foreign key constraint uses the RELY hint, which tells the")
    _say("query planner that the relationship can be trusted for plan")
    _say("optimization (such as eliminating unnecessary joins).")
    _say("")
    _say("Creating fact_orders ...")

    spark.sql(f"DROP TABLE IF EXISTS `{_catalog}`.`{_schema}`.fact_orders")
    spark.sql(f"""
        CREATE TABLE `{_catalog}`.`{_schema}`.fact_orders (
            order_id       STRING,
            customer_key   BIGINT NOT NULL,
            product_key    BIGINT NOT NULL,
            store_key      BIGINT NOT NULL,
            date_key       BIGINT NOT NULL,
            quantity       INT,
            total_amount   DOUBLE,
            CONSTRAINT fk_fact_orders_customer FOREIGN KEY (customer_key)
                REFERENCES `{_catalog}`.`{_schema}`.dim_customer(customer_key) RELY,
            CONSTRAINT fk_fact_orders_product FOREIGN KEY (product_key)
                REFERENCES `{_catalog}`.`{_schema}`.dim_product(product_key) RELY,
            CONSTRAINT fk_fact_orders_store FOREIGN KEY (store_key)
                REFERENCES `{_catalog}`.`{_schema}`.dim_store(store_key) RELY,
            CONSTRAINT fk_fact_orders_date FOREIGN KEY (date_key)
                REFERENCES `{_catalog}`.`{_schema}`.dim_date(date_key) RELY
        )
    """)

    _say("fact_orders table created (empty), with all four FK constraints.")

    # Step 2: insert by joining
    _section("STEP 2 of 3: Insert by joining bronze to the dimensions")
    _say("Every row in orders_bronze carries natural keys (customer_id,")
    _say("product_id, etc.). The fact table wants surrogate keys instead.")
    _say("To translate, we join orders_bronze to each dimension on the")
    _say("natural key and pull the surrogate key out.")
    _say("")
    _say("This is one big INSERT with four JOINs:")
    _say("")
    _say("  orders_bronze.customer_id  ->  dim_customer.customer_key")
    _say("  orders_bronze.product_id   ->  dim_product.product_key")
    _say("  orders_bronze.store_id     ->  dim_store.store_key")
    _say("  orders_bronze.order_date   ->  dim_date.date_key")
    _say("")
    _say("Inserting rows. This may take a few seconds ...")

    spark.sql(f"""
        INSERT INTO `{_catalog}`.`{_schema}`.fact_orders
            (order_id, customer_key, product_key, store_key, date_key,
             quantity, total_amount)
        SELECT
            ob.order_id,
            dc.customer_key,
            dp.product_key,
            ds.store_key,
            dd.date_key,
            ob.quantity,
            ob.total_amount
        FROM `{_catalog}`.`{_schema}`.orders_bronze ob
        JOIN `{_catalog}`.`{_schema}`.dim_customer dc ON ob.customer_id = dc.customer_id
        JOIN `{_catalog}`.`{_schema}`.dim_product  dp ON ob.product_id  = dp.product_id
        JOIN `{_catalog}`.`{_schema}`.dim_store    ds ON ob.store_id    = ds.store_id
        JOIN `{_catalog}`.`{_schema}`.dim_date     dd ON ob.order_date  = dd.full_date
    """)

    n_fact = spark.sql(
        f"SELECT COUNT(*) AS n FROM `{_catalog}`.`{_schema}`.fact_orders"
    ).collect()[0]["n"]
    _say(f"Inserted {n_fact:,} rows into fact_orders.")

    # Step 3: verify
    _section("STEP 3 of 3: Verify referential integrity")
    _say("If the four joins preserved every row, fact_orders should have the")
    _say("same row count as orders_bronze. If any natural key in orders_bronze")
    _say("did not find a match in its dimension, that row would be dropped")
    _say("and the counts would differ.")

    n_bronze = spark.sql(
        f"SELECT COUNT(*) AS n FROM `{_catalog}`.`{_schema}`.orders_bronze"
    ).collect()[0]["n"]
    _say("")
    _say(f"  orders_bronze rows  {n_bronze:>10,}")
    _say(f"  fact_orders rows    {n_fact:>10,}")
    _say("")
    if n_bronze == n_fact:
        _say("The counts match. Every row in orders_bronze found a matching")
        _say("customer, product, store, and date in its respective dimension.")
        _say("The Vintage Audio dataset was generated with full referential")
        _say("integrity, so this is the expected result.")
    else:
        _say(f"WARNING: counts do not match ({n_bronze - n_fact:,} row")
        _say("difference). One or more joins dropped rows, which suggests a")
        _say("natural key mismatch somewhere.")

    # Finish
    _section("STAR SCHEMA COMPLETE")
    _say("Schema status:")
    _say("")
    _say("  dim_customer  built  (with primary key)")
    _say("  dim_product   built  (with primary key)")
    _say("  dim_store     built  (with primary key)")
    _say("  dim_date      built  (with primary key)")
    _say("  fact_orders   built  (with four foreign keys, all RELY)")
    _say("")
    _say("Continue scrolling to inspect the model and run an analytical query.")